# Lesson 21 · Diffusion & flow matching: learn to remove noise, then turn noise into new digits

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/21-diffusion-en.ipynb)

This notebook accompanies [Lesson 21 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/diffusion/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/21-diffusion.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
We write a **flow matching** model ourselves in numpy: add noise to the data with a formula, teach a small network to
predict the direction that removes the noise, then walk back from pure noise to the data. First on a ring of 8
clusters of 2-D points, where the best denoiser can be computed exactly; then on the 16-number codes of Lesson 19's
autoencoder, just as Stable Diffusion does.

**What you'll do:**
1. Add noise with a closed-form formula $x_t = (1 - t)\,x_0 + t\,\varepsilon$, and compute the optimal denoiser of the
   ring of 8 clusters: its loss is not 0.
2. Write the backpropagation of a residual MLP by hand, train it with flow matching, then generate samples with Euler
   steps.
3. Check that predicting the noise, predicting the image and predicting the velocity are interchangeable, and that a
   DDIM step is exactly an Euler step: the number of steps is a choice of the solver, not of the model. Then
   straighten the paths with reflow so that one step is enough.
4. Run the flow-matching network and the decoder that the demo uses to generate MNIST digits, and read the results of
   the full sweep (FD, recall, copying).
5. See that the optimal denoiser only copies the training images, and that classifier-free guidance trades diversity
   for adherence.

**Background:** [Lesson 19 · Autoencoders & VAEs](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/autoencoder-vae/)
(the decoder and the 16-dimensional codes are reused here),
[Lesson 20 · GANs — adversarial networks](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gan/) (the FD metric,
precision/recall, the copying test), the residual block of
[Lesson 13 · Skip connections & ResNet](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/resnet/), the Adam optimizer
of [Lesson 07](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/). **Run time:** under a minute on a CPU,
including the first download of ~10 MB of MNIST training images and ~1 MB of the demo's results.

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. The full training
and evaluation runs (20,000 steps × 3 seeds for each configuration, 10,000 samples per cell of the sweep, about an
hour on 12 cores), however, are loaded from the very files the demo uses; the notebook reruns a small version of each
mechanism and checks what can be recomputed.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `fmt()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import math
import time
import urllib.request
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def fmt(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38 (never “−0”). Rounds half up on the number's shortest
    decimal form, as the website does (Intl.NumberFormat): 3.55 → 3.6, whereas Python's round(3.55, 1) gives 3.5."""
    q = Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-digits), ROUND_HALF_UP)
    s = f"{float(q) + 0.0:,.{digits}f}"
    return s.replace("-", "−")


def pct(p, digits=2):
    return fmt(100 * p, digits) + "%"


def span(values, digits=1, scale=1.0, unit=""):
    """The min–max range across seeds, as in the demo: 3.4–3.6 (a single number when both ends round the same)."""
    lo, hi = scale * min(values), scale * max(values)
    if fmt(lo, digits) == fmt(hi, digits):
        return fmt(lo, digits) + unit
    return f"{fmt(lo, digits)}–{fmt(hi, digits)}{unit}"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A very small or very large number: 1.9·10⁻⁷."""
    e = math.floor(math.log10(abs(x)))
    return f"{fmt(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return fmt(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """A log-axis tick label: 0.5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Put the labels of every axis of the figure (colorbars included) in the English format; call right before
    plt.show(). A log axis gets 1–2–5 ticks if it is short, or powers of 10 (evenly spaced, at most ~9 ticks) if it
    is long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def tile(images, rows, cols, gap=2):
    """Join rows × cols 28×28 images into one large image, with a white background between the tiles."""
    out = np.zeros((rows * (28 + gap) - gap, cols * (28 + gap) - gap))
    for i, im in enumerate(images[: rows * cols]):
        r, c = divmod(i, cols)
        out[r * (28 + gap) : r * (28 + gap) + 28, c * (28 + gap) : c * (28 + gap) + 28] = np.reshape(im, (28, 28))
    return out


def show_tile(ax, images, rows, cols, title=None):
    ax.imshow(tile(images, rows, cols), cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


print(fmt(13007, 0), fmt(0.1, 1), fmt(-4.3805, 4), pct(0.993527), span([3.55, 3.44, 3.35]), sci(4.04e-5))

## 2 · The data

The ring of 8 clusters in 2-D (sections 3–9) makes its own data. The MNIST part needs:
- **the MNIST training images** (LeCun et al., 1998): 60,000 grayscale 28×28 images with their labels. The cell below
  downloads only the two training files and checks their MD5 hashes; we use the first 1,000 images;
- five results files that the demo uses (`fetch_json`, with an MD5 check):
  - `diffusion-net.json`: the two flow-matching networks that run in the browser (H = 256, 40,000 steps, seed 0; one
    of them conditioned on the label), with the matrices stored as int8 with one float16 scale per column;
  - `diffusion-dec.json`: the 16 → 128 → 256 → 784 decoder of Lesson 19's deep autoencoder (seed 0), compressed the
    same way;
  - `diffusion-mnist.json`: the results of the full sweep (every solver × number of steps, guidance, the baselines,
    the copy rates), and the codes of the first 1,000 training images;
  - `diffusion-2d.json`, `diffusion-reflow.json`: the tables of the ring of 8 clusters and the 2-D reflow network.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes (only the two training files)
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


# Results of the long training runs (the same files as the demo on the web). build.py replaces COURSE_DATA_URL with
# the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
NETS = fetch_json("diffusion-net.json", "cc13e79862e744dc54a7d33815298605")
DEC = fetch_json("diffusion-dec.json", "b4e6a6cc2b2748aeeef7aaa681f919b9")
MN = fetch_json("diffusion-mnist.json", "cea01777115494363a7d2151b16bbc1a")
RING2D = fetch_json("diffusion-2d.json", "d14d58d79a9f2f268e92fc31a677974b")
REFLOW = fetch_json("diffusion-reflow.json", "c1b5b583aaa2ddc9b699fb735ef60a21")
names = [
    "diffusion-net.json",
    "diffusion-dec.json",
    "diffusion-mnist.json",
    "diffusion-2d.json",
    "diffusion-reflow.json",
]
print(f"the demo's 5 files: {fmt(sum((Path('data') / f).stat().st_size for f in names) / 1e6, 1)} MB")

The matrices of the two flow-matching networks are int8 integers multiplied by one float16 scale per column, and the
biases are float16; the decoder is compressed as in Lesson 19 (float16 hidden layers, an int8 output-layer matrix).
The cell below unpacks them into float64, reads the first 1,000 MNIST images and their 16-number codes.

The codes here have been **divided by the standard deviation** of all the training codes (`scale` = 1/standard
deviation), so each number has a scale of about 1, the same scale as the noise $\varepsilon \sim \mathcal N(0, I)$.
Stable Diffusion does exactly this, with a factor of its own (Lesson 19). To decode, do the reverse: divide the code
by `scale`.

In [ ]:
def b64(s, dtype):
    """Decode an array that the demo stores as base64 (little-endian)."""
    return np.frombuffer(base64.b64decode(s), dtype)


def unpack(t):
    if "f16" in t:
        return b64(t["f16"], "<f2").astype(np.float64).reshape(t["shape"])
    if "f32" in t:
        return b64(t["f32"], "<f4").astype(np.float64).reshape(t["shape"])
    scale = b64(t["scale"], "<f2").astype(np.float64)  # one scale per column
    return b64(t["i8"], np.int8).astype(np.float64).reshape(t["shape"]) * scale


with gzip.open(DATA / "train-images-idx3-ubyte.gz") as f:
    X_img = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)[:1000] / 255
with gzip.open(DATA / "train-labels-idx1-ubyte.gz") as f:
    y_all = np.frombuffer(f.read(), np.uint8, offset=8)
y_img = y_all[:1000].astype(int)
print(f"MNIST training set: {fmt(len(y_all), 0)} images; we use the first {fmt(len(X_img), 0)} images")

SCALE = NETS["scale"]
dec = {k: unpack(t) for k, t in DEC["dec"].items()}
Zc = b64(MN["memo"]["codes"], "<f2").astype(np.float64).reshape(-1, 16)  # the (divided) codes of the first 1,000 images


def decode(Z):
    """Divided code → image E[x|z]: Lesson 19's 16 → 128 → 256 → 784 decoder (ReLU, sigmoid at the output layer)."""
    h = np.maximum(np.atleast_2d(Z) / SCALE @ dec["d0.W"] + dec["d0.b"], 0)
    h = np.maximum(h @ dec["d1.W"] + dec["d1.b"], 0)
    return 0.5 * (1 + np.tanh(0.5 * (h @ dec["out.W"] + dec["out.b"])))


def nearest(A, B):
    """The distance from each row of A to the nearest row of B, and the index of that row."""
    d2 = np.maximum((A**2).sum(1)[:, None] - 2 * A @ B.T + (B**2).sum(1)[None], 0)
    i = d2.argmin(1)
    return np.sqrt(d2[np.arange(len(A)), i]), i


R_img = decode(Zc)
own = (nearest(R_img, X_img)[1] == np.arange(1000)).mean()
print(f"codes: {Zc.shape}, standard deviation {fmt(Zc.std(), 3)} · scale = {fmt(SCALE, 4)}")
print(f"the image rebuilt from a code is nearest to its own original in {pct(own, 1)} of cases; "
      f"average squared error {fmt(((R_img - X_img) ** 2).sum(1).mean())} per image")  # fmt: skip
assert own > 0.99, "the codes don't match the first 1,000 training images: check the data again"

fig, axes = plt.subplots(2, 1, figsize=(10, 2.5))
show_tile(axes[0], X_img[:12], 1, 12, "MNIST training images")
show_tile(axes[1], R_img[:12], 1, 12, "decoded from each image's 16 numbers (Lesson 19's decoder)")
plt.tight_layout()
plt.show()

The top row shows the original images, the bottom row the images decoded from their 16-number codes: slightly
blurry, but the right digit and the right handwriting. Each rebuilt image is closer to its own original than to any
other of the 1,000 images, so the codes really are those of these images.

From section 10 on, the diffusion model works only with the 16-number codes; the fixed decoder draws them as images.
That is the **latent diffusion** of Stable Diffusion, in miniature.

## 3 · The forward process: mixing data with noise

The toy data is the demo's ring of 8 clusters: 8 Gaussian clusters spaced evenly on a circle of radius 2, each with
standard deviation $s = 0.1$. The forward process of flow matching (in the convention of Stable Diffusion 3) mixes a
clean point $x_0$ with noise $\varepsilon \sim \mathcal N(0, I)$:

$$x_t = (1 - t)\,x_0 + t\,\varepsilon, \qquad t \in [0, 1].$$

$t = 0$ is the clean data, $t = 1$ pure noise. There is nothing to learn here, and nothing to simulate step by step
either: to get a point at any noise level, we compute it directly with the formula. The figure below draws the same
2,000 points at five levels of $t$ (orange squares) and the centers of the 8 clusters (blue circles).

In [ ]:
K, RADIUS, S_RING = 8, 2.0, 0.1
MEANS = RADIUS * np.array([[np.cos(2 * np.pi * k / K), np.sin(2 * np.pi * k / K)] for k in range(K)])


def ring_data(n, s=S_RING, seed=0):
    rng = np.random.default_rng(seed)
    return MEANS[rng.integers(0, K, n)] + s * rng.standard_normal((n, 2))


x0_demo = ring_data(2000)
eps_demo = np.random.default_rng(1).standard_normal((2000, 2))
T_SHOW = [0, 0.25, 0.5, 0.75, 1]
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5), sharex=True, sharey=True)
for ax, t in zip(axes, T_SHOW):
    xt = (1 - t) * x0_demo + t * eps_demo
    ax.scatter(*xt.T, s=2, marker="s", color=ORANGE, alpha=0.5, lw=0)
    ax.scatter(*MEANS.T, s=45, facecolors="none", edgecolors=BLUE, lw=1.5)
    ax.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), aspect="equal", title=f"t = {fmt(t, 2)}")
plt.tight_layout()
fmt_axes(fig)
plt.show()
sd_t = [((1 - t) * x0_demo + t * eps_demo).std(0).mean() for t in T_SHOW]
print(
    "standard deviation of xₜ along each axis:", " · ".join(f"t = {fmt(t, 2)}: {fmt(v)}" for t, v in zip(T_SHOW, sd_t))
)

By $t = 0.5$ the clusters have merged; at $t = 1$ no trace of the data is left. One detail: the cloud halfway along
is **narrower** than at the two ends (standard deviation 0.84 at $t = 0.75$, versus 1.41 at $t = 0$ and 1.00 at
$t = 1$), because the variance is $(1 - t)^2 \operatorname{Var}(x_0) + t^2$. The path of flow matching is a straight
segment between two points; it does not keep the variance constant the way DDPM's schedule does (section 8).

## 4 · The best denoiser, computed exactly

The network will learn the **velocity** $v = \varepsilon - x_0$ (the derivative of $x_t$ with respect to $t$) from
$x_t$ and $t$ alone. But looking at $x_t$, we cannot tell which pair $(x_0, \varepsilon)$ it came from. The best
prediction under squared error is the conditional mean:

$$v^*(x_t, t) = \mathbb E[\varepsilon - x_0 \mid x_t] = \frac{x_t - \mathbb E[x_0 \mid x_t]}{t}.$$

For a Gaussian mixture, $\mathbb E[x_0 \mid x_t]$ can be computed exactly. If $x_0$ belongs to cluster $k$ (center
$m_k$), then $x_t \sim \mathcal N\big((1-t)\,m_k,\ \sigma_t^2 I\big)$ with $\sigma_t^2 = (1-t)^2 s^2 + t^2$. The
posterior probability of each cluster is a softmax of $-\lVert x_t - (1-t)m_k \rVert^2 / 2\sigma_t^2$, and within
that cluster $\mathbb E[x_0 \mid x_t, k] = m_k + \frac{(1-t)s^2}{\sigma_t^2}\big(x_t - (1-t)m_k\big)$. The function
below is written for the general form $x_t = a\,x_0 + b\,\varepsilon$, so that section 8 can reuse it with DDPM's
schedule.

In [ ]:
def ring_x0(x, a, b, s=S_RING):
    """E[x₀ | xₜ] for xₜ = a·x₀ + b·ε, x₀ ~ (1/K) Σ N(m_k, s² I): the average of the centers, weighted by the posterior.
    a and b are numbers or arrays (one value per point)."""
    a = np.broadcast_to(np.asarray(a, float), (len(x),))[:, None, None]
    b = np.broadcast_to(np.asarray(b, float), (len(x),))[:, None, None]
    var = a * a * s * s + b * b  # the variance of xₜ within each cluster
    diff = x[:, None, :] - a * MEANS[None]  # (n, K, 2)
    lw = -(diff**2).sum(-1) / (2 * var[:, :, 0])
    w = np.exp(lw - lw.max(1, keepdims=True))
    w /= w.sum(1, keepdims=True)  # the posterior probability of each cluster
    return (w[:, :, None] * (MEANS[None] + (a * s * s / var) * diff)).sum(1)


def ring_v(x, t, s=S_RING):
    """The optimal velocity v*(xₜ, t) = (xₜ − E[x₀|xₜ]) / t of flow matching (t is a number or an array)."""
    tc = np.broadcast_to(np.asarray(t, float), (len(x),))[:, None]
    return (x - ring_x0(x, 1 - tc[:, 0], tc[:, 0], s)) / tc

### 🤔 Predict first
What flow-matching loss $\mathbb E\,\lVert v^*(x_t, t) - (\varepsilon - x_0)\rVert^2$ (with $t$ uniform on $[0, 1]$)
does a perfect network, one exactly equal to $v^*$, reach? Is it 0?

In [ ]:
rng = np.random.default_rng(2)
n_mc = 200_000
x0_mc, e_mc, t_mc = ring_data(n_mc, seed=3), rng.standard_normal((n_mc, 2)), rng.random(n_mc)
xt_mc = (1 - t_mc[:, None]) * x0_mc + t_mc[:, None] * e_mc
target = e_mc - x0_mc
floor = ((ring_v(xt_mc, t_mc) - target) ** 2).sum(1).mean()
print(f"flow-matching loss of the optimal denoiser: {fmt(floor, 2)} (averaged over {fmt(n_mc, 0)} triples (x₀, ε, t))")
print(f"predicting one constant for every point (the mean of ε − x₀): {fmt(target.var(0).sum(), 2)}")
assert floor > 3, "the optimal denoiser cannot have a loss near 0"

Not 0, but about 3.6: most of the target $\varepsilon - x_0$ is **unpredictable** from $x_t$. Each point $x_t$ can
come from a great many (data point, noise) pairs; the network can only learn their average. That is why the loss of
diffusion never goes to 0, and why the loss value cannot be used to compare two models that choose $t$ differently.

The predictable part depends on $t$. The cell below measures, at each $t$, the fraction of the target's variance that
$v^*$ explains (blue line, round dots), and compares it with the 8 values the demo computes in JavaScript (orange
squares).

In [ ]:
T_GRID = np.round(np.arange(0.05, 1.0, 0.05), 2)
x0_s, e_s, g_s = x0_mc[:20000], e_mc[:20000], target[:20000]
share = np.array(
    [1 - ((ring_v((1 - t) * x0_s + t * e_s, t) - g_s) ** 2).sum() / ((g_s - g_s.mean(0)) ** 2).sum() for t in T_GRID]
)
demo_share = {e["t"]: e["share"] for e in RING2D["table"]["0.1"]["explained"]}
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(T_GRID, 100 * share, color=BLUE, marker="o", ms=4, label="notebook (20,000 points per t)")
ax.scatter(list(demo_share), [100 * v for v in demo_share.values()], marker="s", color=ORANGE, zorder=3, label="demo")
ax.set(xlabel="t", ylabel="explained variance (%)", ylim=(0, 100), title="How much of ε − x₀ can v* predict?")
ax.legend(frameon=False)
fmt_axes(fig)
plt.show()
print("  ".join(f"t = {fmt(t, 2)}: {pct(s, 0)}" for t, s in zip(T_GRID, share) if t in demo_share))
assert abs(share[T_GRID == 0.5][0] - demo_share[0.5]) < 0.02, "differs from the demo: check the denoiser again"

Near $t = 0$, $x_t$ is almost the data, so knowing $x_t$ means knowing which cluster $x_0$ belongs to: $v^*$ explains
most of the target. At $t = 0.5$ only 26% is left, and around $t = 0.7$ almost 0: the clusters have merged, and the
optimal velocity is only a blurred average of 8 directions. Near $t = 1$, $x_t \approx \varepsilon$, so the
$\varepsilon$ part of the target becomes predictable again.

## 5 · Sampling: walking back from noise with Euler steps

Sampling means solving the differential equation $\frac{dx}{dt} = v(x, t)$ backward from $t = 1$ (pure noise) to
$t = 0$. The simplest way is **Euler** steps of size $\Delta t$:

$$x_{t - \Delta t} = x_t - \Delta t\; v(x_t, t).$$

Each step is one network call. To compare with the demo, we take exactly the noise the demo uses: the `mulberry32`
random number generator that the demo runs in the browser (rewritten in numpy, matching bit for bit as in Lesson 09)
and the Box–Muller transform. The demo's metrics: the fraction of samples that lie in a cluster (closer than $3s$ to
the nearest center), the number of clusters with at least 1% of the samples, and the mean radius (data: 2).

In [ ]:
def mulberry32(seed, n):
    """The first n 32-bit integers of mulberry32(seed), the generator the demo (JavaScript) uses, in vectorized form."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)  # uint32 multiplication wraps around mod 2³² by itself, like Math.imul
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return t ^ (t >> 14)


def js_normals(seed, n):
    """The demo's normals(mulberry32(seed), n): Box–Muller on each pair of uniform numbers."""
    u = mulberry32(seed, n) / 2**32
    r, a = np.sqrt(-2 * np.log(1 - u[0::2])), 2 * np.pi * u[1::2]
    return np.stack([r * np.cos(a), r * np.sin(a)], 1).ravel()


def grid(k):
    return np.linspace(1, 0, k + 1)  # k equal steps in t, from 1 down to 0


def euler(v, z, ts, path=None):
    x = z.copy()
    for a, b in itertools.pairwise(ts):
        x = x - (a - b) * v(x, a)
        if path is not None:
            path.append(x)
    return x


def ring_quality(x, s=S_RING):
    d = np.sqrt(((x[:, None] - MEANS[None]) ** 2).sum(-1))
    near = d.min(1) < 3 * s
    counts = np.bincount(d.argmin(1)[near], minlength=K)
    return near.mean(), int((counts >= 0.01 * len(x)).sum()), np.hypot(*x.T).mean()


Z_DEMO = [js_normals(seed, 4000).reshape(-1, 2) for seed in (1, 2, 3)]  # the demo's 3 seeds × 2,000 noise points
print("the first 4 uniform numbers of mulberry32(1):", [fmt(u, 6) for u in mulberry32(1, 4) / 2**32])
assert mulberry32(1, 1)[0] / 2**32 == 0.6270739405881613  # the value reel/diffusion/diff.py checks against JavaScript

### 🤔 Predict first
Use the optimal denoiser itself (no learning error at all) and just **one** Euler step from $t = 1$ to $t = 0$.
Where will the 2,000 noise points land? What percentage of them will lie in a cluster?

In [ ]:
x_one = euler(ring_v, Z_DEMO[0], grid(1))
q_one = ring_quality(x_one)
print(f"after 1 step: {pct(q_one[0], 0)} of the samples in a cluster, {q_one[1]}/8 clusters have samples, "
      f"the point farthest from the origin is {fmt(np.hypot(*x_one.T).max(), 6)} away from it")  # fmt: skip
print("mean of the data:", [fmt(m, 6) for m in MEANS.mean(0)])
assert q_one[0] == 0 and np.hypot(*x_one.T).max() < 1e-9

Every point lands exactly at the origin, the mean of the data, where there is no cluster. At $t = 1$, $x_t$ is pure
noise and so carries no information about $x_0$: the best prediction $\mathbb E[x_0 \mid x_1]$ is the mean of **all**
the data points, and one Euler step goes straight there. Each (noise, data) pair is joined by a straight segment, but
the optimal velocity field is the average of all the segments passing through $x_t$, and that average is **curved**.
Following a curve takes many steps.

The cell below runs 1, 2, 4, 8, 32 and 100 Euler steps on the demo's 3 noise sets, and compares the results with the
table the demo precomputed.

In [ ]:
STEPS = [1, 2, 4, 8, 32, 100]
table = RING2D["table"]["0.1"]["rows"]
for k in STEPS:
    qs = [ring_quality(euler(ring_v, z, grid(k))) for z in Z_DEMO]
    stored = [r["inMode"] for r in table[f"euler/{k}"]]
    print(f"{k:>5} {'step: ' if k == 1 else 'steps:'} in a cluster {span([q[0] for q in qs], 1, 100, '%'):>12}, "
          f"{span([q[1] for q in qs], 0)}/8 clusters, radius {span([q[2] for q in qs], 2)} "
          f"· demo: {span(stored, 1, 100, '%')}")  # fmt: skip
    assert np.allclose([q[0] for q in qs], stored, atol=1e-3), "differs from the demo: check the solver again"

The denoiser is optimal, so every error here is the **solver**'s. 2 steps put only 4.2–4.6% of the samples in a
cluster, 4 steps already 76.8–78.4%, 8 steps 97.3–98.2%, and beyond that more steps hardly change anything. The
figure below draws the paths of 48 noise points with 4 and 32 steps (gray), the endpoints (orange squares) and the
cluster centers (blue circles). With 32 steps the paths are clearly curved: at first every point is pulled toward the
origin together, and only halfway along do they split off toward their own clusters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, k in zip(axes, (4, 32)):
    path = [Z_DEMO[0]]
    xk = euler(ring_v, Z_DEMO[0], grid(k), path)
    P = np.array(path)[:, :48]  # (step, point, 2)
    for j in range(48):
        ax.plot(P[:, j, 0], P[:, j, 1], color=GREY, lw=0.7, alpha=0.7)
    ax.scatter(*xk.T, s=3, marker="s", color=ORANGE, alpha=0.6, lw=0)
    ax.scatter(*MEANS.T, s=60, facecolors="none", edgecolors=BLUE, lw=1.5)
    ax.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), aspect="equal", xlabel="x₁")
    ax.set_title(f"{k} Euler steps: {pct(ring_quality(xk)[0], 0)} of the samples in a cluster", fontsize=10)
axes[0].set_ylabel("x₂")
plt.tight_layout()
fmt_axes(fig)
plt.show()

z_path = js_normals(4, 1000).reshape(-1, 2)  # 500 points, as in the demo
path = [z_path]
x_end = euler(ring_v, z_path, grid(1000), path)
length = np.hypot(*np.diff(np.array(path), axis=0).transpose(2, 0, 1)).sum(0)
chord = np.hypot(*(x_end - z_path).T)
ratios = np.sort(length[chord > 1e-9] / chord[chord > 1e-9])
ratio = ratios[len(ratios) // 2]  # the median the demo's way (the middle element of the sorted array)
print(
    f"the actual path (1,000 steps) is {fmt(ratio, 2)} times as long as the segment joining its start and end (median)"
)
assert abs(ratio - RING2D["table"]["0.1"]["pathRatio"]) < 1e-3, "differs from the demo"

The median path is 1.5 times as long as its chord. This curvature is why one step is not enough (section 9
straightens it out).

## 6 · Training a flow-matching network

The exact denoiser is available only when we know the distribution of the data. Usually we have only samples, so we
learn $v_\theta$ by regression:

$$\mathcal L = \mathbb E_{x_0,\ \varepsilon,\ t \sim \mathcal U(0, 1)}\ \big\lVert v_\theta(x_t, t) - (\varepsilon - x_0) \big\rVert^2.$$

The network is identical to the demo's (`reel/diffusion/diff.py`): a residual MLP in the row convention
$h = f(xW + b)$. The input is $x_t$ concatenated with 17 features of $t$ ($t$ itself, and the $\sin$ and $\cos$ of
$2^k \pi t$ for $k < 8$), then $h_1 = \mathrm{SiLU}(x W_0 + b_0)$, two residual blocks
$h \leftarrow h + \mathrm{SiLU}(h W_i + b_i)$ (Lesson 13), and a linear output layer $W_o$ initialized to 0.
$\mathrm{SiLU}(a) = a\,\sigma(a)$, with derivative $\sigma(a)\,(1 + a(1 - \sigma(a)))$. The conditional network
(section 12) also concatenates an 11-dimensional one-hot vector for the label (the 10 digits and "no label" ∅).

In [ ]:
def temb(t, n):
    """17 features of t: t, sin(2ᵏπt), cos(2ᵏπt) for k = 0…7."""
    t = np.broadcast_to(np.asarray(t, float), (n,))[:, None]
    a = t * (2.0 ** np.arange(8) * np.pi)[None]
    return np.concatenate([t, np.sin(a), np.cos(a)], 1)


def inputs(x, t, y=None):
    parts = [x, temb(t, len(x))]
    if y is not None:  # label 0–9, or 10 = ∅
        parts.append(np.eye(11)[y])
    return np.concatenate(parts, 1)


def sigmoid(a):
    return 0.5 * (1 + np.tanh(0.5 * a))


def init(din, dout, H, seed, L=3):
    rng = np.random.default_rng(seed)
    p = {"W0": rng.standard_normal((din, H)) * np.sqrt(2 / din), "b0": np.zeros(H)}
    for i in range(1, L):
        p[f"W{i}"], p[f"b{i}"] = rng.standard_normal((H, H)) * np.sqrt(1 / H), np.zeros(H)
    p["Wo"], p["bo"] = np.zeros((H, dout)), np.zeros(dout)
    return p


def forward(p, X, keep=False):
    a = X @ p["W0"] + p["b0"]
    s = sigmoid(a)
    h, cache = a * s, [(X, a, s)]
    for i in range(1, sum(k[0] == "W" for k in p) - 1):  # the residual blocks W1, W2, …
        a = h @ p[f"W{i}"] + p[f"b{i}"]
        s = sigmoid(a)
        cache.append((h, a, s))
        h = h + a * s
    out = h @ p["Wo"] + p["bo"]
    return (out, (cache, h)) if keep else out


def backward(p, cache, dout):
    cache, h = cache
    g = {"Wo": h.T @ dout, "bo": dout.sum(0)}
    dh = dout @ p["Wo"].T
    for i in range(len(cache) - 1, -1, -1):
        hin, a, s = cache[i]
        da = dh * s * (1 + a * (1 - s))  # the derivative of SiLU
        g[f"W{i}"], g[f"b{i}"] = hin.T @ da, da.sum(0)
        dh = (dh if i > 0 else 0) + da @ p[f"W{i}"].T  # residual block: the gradient also flows through the shortcut
    return g

Before trusting the hand-written backpropagation, we compare it with finite differences on a small network with
random weights (with $W_o$ nonzero too, so that the gradient flows through every layer):

In [ ]:
rng = np.random.default_rng(0)
p_chk = {k: v + 0.3 * rng.standard_normal(v.shape) for k, v in init(5, 3, 6, 1).items()}
X_chk, tgt_chk = rng.standard_normal((4, 5)), rng.standard_normal((4, 3))
out, cache = forward(p_chk, X_chk, keep=True)
g = backward(p_chk, cache, out - tgt_chk)  # the gradient of ½‖out − tgt‖²
worst = 0.0
for k, w in p_chk.items():
    for i in range(w.size):
        old = w.flat[i]
        w.flat[i] = old + 1e-6
        lp = 0.5 * ((forward(p_chk, X_chk) - tgt_chk) ** 2).sum()
        w.flat[i] = old - 1e-6
        lm = 0.5 * ((forward(p_chk, X_chk) - tgt_chk) ** 2).sum()
        w.flat[i] = old
        num = (lp - lm) / 2e-6
        worst = max(worst, abs(num - g[k].flat[i]) / max(1e-8, abs(num) + abs(g[k].flat[i])))
print(f"{sum(w.size for w in p_chk.values())} parameters, largest relative error {sci(worst)}")
assert worst < 1e-6

The backpropagation matches the finite differences. The training loop below follows the demo: each step takes 256
data points, 256 fresh noise samples and 256 uniform values of $t$, computes $x_t$ with the formula of section 3, then
takes one Adam step (Lesson 07) on the squared error against the target $\varepsilon - x_0$. The learning rate ramps
up over the first 2% of the steps, then decays along a cosine. The model used for sampling is the **moving average
(EMA)** of the weights, as in every real diffusion model. Pass `E` to pair each data point with a fixed noise sample in
advance (section 9).

In [ ]:
def lr_at(s, lr, steps):
    warm = max(1, steps // 50)
    return lr * (s + 1) / warm if s < warm else lr * 0.5 * (1 + np.cos(np.pi * (s - warm) / (steps - warm)))


def train_fm(X, seed=0, E=None, H=64, steps=3000, lr=5e-3, batch=256, ema_decay=0.995):
    """Flow matching on the rows of X: returns the EMA weights and the loss of each step."""
    rng = np.random.default_rng(seed)
    d = X.shape[1]
    p = init(d + 17, d, H, seed)
    m1, m2 = {k: np.zeros_like(v) for k, v in p.items()}, {k: np.zeros_like(v) for k, v in p.items()}
    ema, losses = {k: v.copy() for k, v in p.items()}, []
    for s in range(steps):
        idx = rng.integers(0, len(X), batch)
        x0, t = X[idx], rng.random(batch)
        e = E[idx] if E is not None else rng.standard_normal((batch, d))
        xt = (1 - t[:, None]) * x0 + t[:, None] * e
        out, cache = forward(p, inputs(xt, t), keep=True)
        diff = out - (e - x0)
        losses.append((diff**2).sum(1).mean())
        g = backward(p, cache, (2 / diff.size) * diff)
        c1, c2, lr_s = 1 - 0.9 ** (s + 1), 1 - 0.999 ** (s + 1), lr_at(s, lr, steps)
        for k in p:  # Adam, then the moving average of the weights
            m1[k] = 0.9 * m1[k] + 0.1 * g[k]
            m2[k] = 0.999 * m2[k] + 0.001 * g[k] ** 2
            p[k] -= lr_s * (m1[k] / c1) / (np.sqrt(m2[k] / c2) + 1e-8)
            ema[k] = ema_decay * ema[k] + (1 - ema_decay) * p[k]
    return ema, np.array(losses)


def net_v(p, y=None):
    """The velocity of a network: v(x, t)."""
    return lambda x, t: forward(p, inputs(x, t, None if y is None else y))


t0 = time.perf_counter()
X_ring = ring_data(20000, seed=10)
p_ring, loss_ring = train_fm(X_ring)
print(f"3,000 steps ({fmt(time.perf_counter() - t0, 1)} seconds): loss over the last 500 steps {fmt(loss_ring[-500:].mean(), 2)}, "
      f"floor of the optimal denoiser {fmt(floor, 2)}")  # fmt: skip
assert loss_ring[-500:].mean() - floor < 0.15, "the network hasn't finished learning: check the training loop again"

The loss settles just above the 3.59 floor of section 4. We sample from the network with the same noise as the demo:

In [ ]:
Z_EVAL = np.random.default_rng(9).standard_normal((2000, 2))  # the 2,000 noise samples the demo's reflow table uses
fm_rows = [r for r in REFLOW["rows"] if r["kind"] == "fm" and r["s"] == S_RING]
live_fm = {}
print("  steps │ network just trained │ optimal denoiser │ demo: same kind of network, 12,000 steps, 3 seeds")
for k in (1, 2, 4, 8, 32):
    live_fm[k] = ring_quality(euler(net_v(p_ring), Z_EVAL, grid(k)))[0]
    exact_k = ring_quality(euler(ring_v, Z_EVAL, grid(k)))[0]
    print(f"{k:>7} │ {pct(live_fm[k], 1):>20} │ {pct(exact_k, 1):>16} │ "
          f"{span([r['steps'][str(k)]['inMode'] for r in fm_rows], 1, 100, '%')}")  # fmt: skip
q32 = ring_quality(euler(net_v(p_ring), Z_EVAL, grid(32)))
print(f"32 steps: {q32[1]}/8 clusters have samples, mean radius {fmt(q32[2])}")
assert live_fm[1] < 0.01 and live_fm[32] > 0.8 and q32[1] == 8

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2), sharey=True)
for ax, k in zip(axes, (1, 4, 32)):
    xk = euler(net_v(p_ring), Z_EVAL, grid(k))
    ax.scatter(*xk.T, s=3, marker="s", color=ORANGE, alpha=0.5, lw=0)
    ax.scatter(*MEANS.T, s=60, facecolors="none", edgecolors=BLUE, lw=1.5)
    ax.set(
        xlim=(-3.3, 3.3),
        ylim=(-3.3, 3.3),
        aspect="equal",
        title=f"trained network, {k} {'step' if k == 1 else 'steps'}",
    )
plt.tight_layout()
fmt_axes(fig)
plt.show()

The network, trained in a few seconds, already generates all 8 clusters, and behaves like the optimal denoiser: 1 step
falls to the middle, and the first few steps only pull the samples toward the circle. But it needs more steps to get
properly into the clusters (compare the "network just trained" column with the "optimal denoiser" column at 8 steps),
and its samples are more smeared around each cluster: that is the learning error. The demo's network of the same kind,
trained longer, closes part of that gap (last column).

## 7 · Predicting the noise, the image or the velocity: the same problem

DDPM (Ho et al., 2020) has the network predict the noise $\varepsilon$; some models predict the clean image $x_0$
directly; flow matching predicts the velocity $v = \varepsilon - x_0$. From $x_t = (1 - t)x_0 + t\varepsilon$ and any
one of the three, we can derive the other two:

$$\hat x_0 = x_t - t\,v, \qquad \hat\varepsilon = x_t + (1 - t)\,v, \qquad v = \frac{x_t - \hat x_0}{t} = \frac{\hat\varepsilon - x_t}{1 - t}.$$

So **predicting the noise is also predicting the image**: at every step, the network always implicitly has a clean
image $\hat x_0$ in mind. The cell below checks these identities on random numbers (this is the first identity that
`reel/diffusion/diff.py` checks itself).

In [ ]:
rng = np.random.default_rng(0)
x0_r, e_r, t_r = rng.standard_normal((4, 16)), rng.standard_normal((4, 16)), 0.37
xt_r, v_r = (1 - t_r) * x0_r + t_r * e_r, e_r - x0_r
err_conv = max(np.abs(xt_r - t_r * v_r - x0_r).max(), np.abs(xt_r + (1 - t_r) * v_r - e_r).max(),
               np.abs((xt_r - x0_r) / t_r - v_r).max(), np.abs((e_r - xt_r) / (1 - t_r) - v_r).max())  # fmt: skip
print(f"x̂₀ = xₜ − t·v, ε̂ = xₜ + (1 − t)·v and the two inverse conversions: largest error {sci(err_conv)}")
assert err_conv < 1e-12

The three targets are the same regression problem; converting between them only multiplies the error by a factor
that depends on $t$, that is, it changes the **weight** of each noise level in the loss (Kingma & Gao, 2023). But that
factor is sometimes very large: getting a velocity from a network that predicts $\varepsilon$ requires dividing by
$1 - t$, and from a network that predicts $x_0$, dividing by $t$. The demo trained all three on the same MNIST codes
(H = 512 networks, 20,000 steps, 3 seeds) and measured the velocity error
$\mathbb E\,\lVert \hat v - (\varepsilon - x_0)\rVert^2$ on the validation codes. The cell below only reads those
results:

In [ ]:
def cells(model, sampler, nfe, key="fd", s=None):
    """The stored values (one number per seed) of one cell of the sweep results."""
    rows = [c for c in MN["cells"] if c["model"] == model and c["sampler"] == sampler and c["nfe"] == nfe]
    rows = [c for c in rows if s is None or c.get("s") == s]
    return [c[key] for c in sorted(rows, key=lambda c: c["seed"]) if isinstance(c.get(key), (int, float))]


TS = MN["setup"]["ts"]
TARGET_NAME = {"ae16-fm": "predict v (flow matching)", "ae16-x0": "predict x₀", "ae16-eps": "predict ε"}
param = {m: [r["err"] for r in MN["param"] if r["model"] == m] for m in TARGET_NAME}
print("network                   " + "".join(f"t = {fmt(t, 2):<6}" for t in TS))
for m, name in TARGET_NAME.items():
    print(f"{name:<26}" + "".join(f"{span([e[i] for e in param[m]], 0):<10}" for i in range(len(TS))))
fd_t = {m: [c["fd"] for c in MN["cells"] if c["model"] == m and c["sampler"] == "euler" and c["nfe"] == 100]
        for m in TARGET_NAME}  # fmt: skip
print("FD after 100 Euler steps:", " · ".join(f"{TARGET_NAME[m]} {span(v, 1)}" for m, v in fd_t.items()))
print(f"predict v, t drawn from a logit-normal as in SD3: FD {span(cells('ae16-ln', 'euler', 100))} after 100 steps")

Halfway along, the three networks err equally (within the variation across seeds). At the two ends they don't: the
network that predicts $\varepsilon$ errs more than 7 times as much at $t = 0.98$ (92–97 versus 12–13), because near
pure noise $x_t \approx \varepsilon$, and a small error in $\hat\varepsilon$ gets divided by $1 - t = 0.02$. The
network that predicts $x_0$ has the opposite blind spot, at $t = 0.02$. A large error in the first step spoils the
whole trajectory: the FD of the $\varepsilon$-predicting network is 45–75 after 100 steps. (FD is the Fréchet distance
between the LeNet features of the generated images and those of 10,000 test images, as in Lessons 19–20: the lower,
the better.) Predicting $v$ mixes the two and has no blind spot; Stable Diffusion 3 and FLUX both predict $v$. The SD3
paper also draws $t$ from a logit-normal distribution, which puts more training samples in the middle of the path; on
the MNIST codes, that brings the FD after 100 steps down to 3.0–3.1.

## 8 · DDPM, DDIM and flow matching: different solvers, not different models

DDPM writes the forward process differently: $t$ is an integer from 1 to $T = 1{,}000$, and
$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1 - \bar\alpha_t}\,\varepsilon$ with $\bar\alpha_t = \prod_{i \le t}(1 - \beta_i)$ and
$\beta_t$ increasing linearly from $10^{-4}$ to $0.02$. Ho et al. (2020) write that they chose $T = 1{,}000$ "so that
the number of neural network evaluations needed during sampling matches previous work", and set it "without a sweep".
DDIM (Song et al., 2020) samples deterministically with the DDPM network itself, on a subsequence of the steps: from
$x_t$, predict $\hat x_0$ and $\hat\varepsilon$, then set
$x_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat x_0 + \sqrt{1 - \bar\alpha_{t'}}\,\hat\varepsilon$.

Written on the flow-matching path, the DDIM step from $t$ to $t'$ is $x_{t'} = (1 - t')\,\hat x_0 + t'\,\hat\varepsilon$.
Substituting $\hat\varepsilon = (x_t - (1 - t)\hat x_0)/t$, we get exactly $x_t - (t - t')\,(x_t - \hat x_0)/t$:
**one Euler step**. The cell below checks this with the exact denoiser of the ring of 8 clusters (the second identity
that `diff.py` checks itself), then checks something stronger: dividing DDPM's $x_t$ by
$c_t = \sqrt{\bar\alpha_t} + \sqrt{1 - \bar\alpha_t}$ gives exactly the straight line of flow matching with
$\tau = \sqrt{1 - \bar\alpha_t}/c_t$. So a whole DDIM run on the DDPM schedule is Euler on a different $\tau$ grid.

In [ ]:
BETA = np.linspace(1e-4, 0.02, 1000)
ABAR = np.cumprod(1 - BETA)  # ᾱ_t, t = 1…1,000 at index t − 1


def respace(k):
    """A subsequence of k DDPM indices, evenly spaced from T down to 1 (as in the demo)."""
    return np.unique(np.round(np.linspace(1, 1000, k)).astype(int))[::-1] if k > 1 else np.array([1000])


def ddim_ring(z, k):
    """DDIM with the exact denoiser of the ring, on the DDPM schedule."""
    x, taus = z.copy(), respace(k)
    for i, ti in enumerate(taus):
        ab = ABAR[ti - 1]
        abp = ABAR[taus[i + 1] - 1] if i + 1 < len(taus) else 1.0
        x0 = ring_x0(x, np.sqrt(ab), np.sqrt(1 - ab))
        e = (x - np.sqrt(ab) * x0) / np.sqrt(1 - ab)
        x = np.sqrt(abp) * x0 + np.sqrt(1 - abp) * e
    return x


z = Z_DEMO[0]
t_a, t_b = 0.8, 0.55
x0h = ring_x0(z, 1 - t_a, t_a)
step_euler = z - (t_a - t_b) * (z - x0h) / t_a
step_ddim = (1 - t_b) * x0h + t_b * (z - (1 - t_a) * x0h) / t_a
print(
    f"one Euler step and one DDIM step, t = {fmt(t_a, 2)} → {fmt(t_b, 2)}: difference {sci(np.abs(step_euler - step_ddim).max())}"
)
assert np.abs(step_euler - step_ddim).max() < 1e-12
for k in (4, 8):
    ab = ABAR[respace(k) - 1]
    c = np.sqrt(ab) + np.sqrt(1 - ab)
    tau = np.r_[np.sqrt(1 - ab) / c, 0.0]
    gap = np.abs(ddim_ring(z, k) - euler(ring_v, z / c[0], tau)).max()
    print(f"DDIM with {k} steps = Euler on the grid τ = {[fmt(u, 3) for u in tau]}: difference {sci(gap)}")
    assert gap < 1e-9
print(f"ᾱ_T = {sci(ABAR[-1])}: at step 1,000, x_T still keeps √ᾱ_T = {fmt(np.sqrt(ABAR[-1]), 4)} times the image")

The two solvers agree to within rounding error. So DDPM, DDIM and flow matching are not three kinds of model: a
network that predicts $\varepsilon$, $x_0$ or $v$ is just three ways of writing the same denoiser (section 7), and
DDIM is Euler. What differs is **where the steps are placed**. DDPM's uniform grid with 4 steps is
$\tau$ = 0.994 → 0.905 → 0.594 → 0.010 → 0: the first two steps are almost wasted in the region of pure noise, and
then comes one giant step from 0.594 down to 0.010. The demo measures exactly this on the ring of 8 clusters (the cell
below reads its table, 3 noise seeds):

In [ ]:
for k in (4, 8, 32):
    row = {smp: span([r["inMode"] for r in table[f"{smp}/{k}"]], 1, 100, "%") for smp in ("euler", "ddim", "ddpm")}
    print(f"{k:>2} steps, samples in a cluster: Euler (uniform t grid) {row['euler']:>11} · DDIM (DDPM grid) {row['ddim']:>11} · "
          f"stochastic DDPM {row['ddpm']}")  # fmt: skip

Same perfect denoiser, same 4 steps: the uniform grid in $t$ puts 76.8–78.4% of the samples in a cluster, the DDPM
grid only 1.8–1.9%. With 32 steps every method does well. DDPM's stochastic solver (which adds fresh noise at every
step) is even slightly worse than DDIM with few steps, and only equal or better with many steps.

The same holds on MNIST. The demo trained both a DDPM network ($T = 1{,}000$, predicting $\varepsilon$) and a
flow-matching network on the same codes, 3 seeds each, then measured the FD on 10,000 samples for each solver and
number of network calls:

In [ ]:
LINES = [("ae16-fm", "euler", "flow matching, Euler"), ("ae16-ddpm", "ddim", "DDPM, DDIM solver"),
         ("ae16-ddpm", "ddpm", "DDPM, stochastic")]  # fmt: skip
print("network calls │ " + " │ ".join(f"{name:<20}" for _, _, name in LINES))
for k in (1, 2, 4, 8, 16, 32, 100, 1000):
    print(
        f"{fmt(k, 0):>13} │ "
        + " │ ".join(f"{span(cells(m, smp, k)) if cells(m, smp, k) else '—':<20}" for m, smp, _ in LINES)
    )
print(
    f"1,000 Euler steps (seed 0): FD {fmt(cells('ae16-fm', 'euler', 1000)[0])} · 1/√ᾱ_T = {fmt(1 / np.sqrt(ABAR[-1]), 0)}"
)
for m, smp, name in LINES[::2]:
    print(f"1 call, {name}: standard deviation of the codes {span(cells(m, smp, 1, 'codeStd'))}, "
          f"precision {span(cells(m, smp, 1, 'P'), 2)}, recall {span(cells(m, smp, 1, 'R'), 2)}")  # fmt: skip

Three things to read in the table (a low FD is good):
- At 8 network calls, the same DDPM network gives an FD of 16.1–18.2 with the stochastic solver and 10.4–12.1 with
  DDIM: with few steps, adding fresh noise costs you.
- Flow matching with Euler at 8 calls reaches 6.1–6.6, better than DDIM on the DDPM network. Part of that advantage
  comes from the uniform $t$ grid, not from the training objective (the cell above).
- 32 Euler steps give 3.7–3.9; 1,000 steps (seed 0) give 3.45. The number 1,000 belongs to DDPM's training, not to
  the number of steps needed to generate images.

One step of the DDPM network (FD 58–70) looks "better" than one step of flow matching (247–284), but not because it
is more correct. At $t = T$, the image makes up only $\sqrt{\bar\alpha_T} = 0.0064$ of $x_T$ (the cell above), so
$\hat x_0 = (x_T - \sqrt{1 - \bar\alpha_T}\,\hat\varepsilon)/\sqrt{\bar\alpha_T}$ multiplies the error of
$\hat\varepsilon$ by 157, exactly the blind spot of section 7. The generated codes have a standard deviation of
5.1–5.2, 5 times that of the real codes: the samples are diverse (recall 0.65–0.73) but most of them look like no
digit at all (precision 0.33–0.41). One step of flow matching is the opposite: every sample is the same "average
digit" (precision 0.84–0.99, recall near 0). FD lumps the two into
a single number; both are broken.

## 9 · Straightening the paths: reflow

One Euler step fails because the velocity field is curved (section 5): it is the average of all the segments joining
a **random** noise sample to a **random** data point. Reflow (Liu et al., 2022) changes the pairing: take each noise
sample $z$, run the existing model to find which point $x$ it takes $z$ to, then train flow matching again on those
fixed pairs $(x, z)$. These pairs are matched along the paths of a differential equation, and two such paths never
cross; so the straight segments joining them cross far less often, each $x_t$ belongs almost entirely to a single
pair, and the new velocity field is almost straight.

The cell below does exactly that on the ring of 8 clusters: it pairs 10,000 noise samples with the endpoints of 100
Euler steps on the exact field, then trains the same network with the same recipe as section 6, except that `E` is
fixed.

In [ ]:
t0 = time.perf_counter()
Z_pair = np.random.default_rng(10).standard_normal((10000, 2))
X_pair = euler(ring_v, Z_pair, grid(100))  # the point the exact field takes each noise sample to
p_rf, _ = train_fm(X_pair, E=Z_pair)
print(f"reflow: {fmt(time.perf_counter() - t0, 1)} seconds")


def straightness(v, z, steps=200):
    """The median of (path length) / (chord) over the points z."""
    path = [z]
    x = euler(v, z, grid(steps), path)
    length = np.hypot(*np.diff(np.array(path), axis=0).transpose(2, 0, 1)).sum(0)
    return np.median(length / np.hypot(*(x - z).T))


live_rf = {k: ring_quality(euler(net_v(p_rf), Z_EVAL, grid(k)))[0] for k in (1, 2, 4, 8, 32)}
print("samples in a cluster     │ " + " · ".join(f"{k} {'step' if k == 1 else 'steps'}" for k in live_rf))
print("flow matching (section 6)│ " + " · ".join(pct(live_fm[k], 0) for k in live_rf))
print("reflow                   │ " + " · ".join(pct(live_rf[k], 0) for k in live_rf))
print(f"path / chord: flow matching {fmt(straightness(net_v(p_ring), Z_EVAL[:300]), 2)}, "
      f"reflow {fmt(straightness(net_v(p_rf), Z_EVAL[:300]), 3)}")  # fmt: skip
assert live_rf[1] > 0.5 > live_fm[1]

One step of the reflow network already puts about three quarters of the samples in the right cluster, versus 0% for
the ordinary flow-matching network, and its paths are almost perfectly straight (ratio ≈ 1). The demo uses a reflow
network trained longer, on more pairs; the file `diffusion-reflow.json` contains its float32 weights, so we can run
that very network and compare with the demo's table:

In [ ]:
p_rf_demo = {k: unpack(t) for k, t in REFLOW["net"].items()}
setup = REFLOW["setup"]
print(f"the demo's reflow network: H = {setup['H']}, {fmt(setup['pairs'], 0)} pairs ({setup['coupling']}), "
      f"{fmt(setup['steps'], 0)} training steps")  # fmt: skip
row0 = next(r for r in REFLOW["rows"] if r["kind"] == "rf2" and r["s"] == S_RING and r["seed"] == 0)
rf_rows = [r for r in REFLOW["rows"] if r["kind"] == "rf2" and r["s"] == S_RING]
for k in (1, 2, 4, 8, 16, 32):
    q = ring_quality(euler(net_v(p_rf_demo), Z_EVAL, grid(k)))[0]
    print(f"{k:>2} {'step: ' if k == 1 else 'steps:'} {pct(q, 2)} (demo, seed 0: {pct(row0['steps'][str(k)]['inMode'], 2)}; 3 seeds: "
          f"{span([r['steps'][str(k)]['inMode'] for r in rf_rows], 1, 100, '%')})")  # fmt: skip
    assert abs(q - row0["steps"][str(k)]["inMode"]) < 0.002, "differs from the demo: check the reflow network again"

On MNIST, the demo applies reflow to the flow-matching network of the 16-number codes (one pair per training code,
from 100 Euler steps, 3 seeds):

In [ ]:
for k in (1, 2, 4, 8, 32):
    print(
        f"{k:>2} {'step: ' if k == 1 else 'steps:'} FD flow matching {span(cells('ae16-fm', 'euler', k)):>11} · reflow {span(cells('ae16-rf2', 'euler', k))}"
    )
print(f"1 flow-matching step: LeNet reads {span(cells('ae16-fm', 'euler', 1, 'histMax'), 0, 100, '%')} of the samples "
      f"as the same digit; reflow: {span(cells('ae16-rf2', 'euler', 1, 'histMax'), 0, 100, '%')} (real data: ~10% per digit)")  # fmt: skip
print(f"Lesson 20's GAN: FD {span(MN['ref']['gan'])}")

One step of ordinary flow matching gives an FD of 247–284: every code shrinks toward the mean code (section 10
measures this), and LeNet reads 46–94% of the samples as the same digit. After reflow, one step gives 5.4–5.6, better
than the GAN of Lesson 20 (7.9–8.0). The price: at 32 steps, the reflow network (4.0–4.2) is worse than the original
network (3.7–3.9), partly because it learns from the original network's own samples, errors included. The models that
generate images in 1–4 steps (SDXL Turbo, FLUX.1 [schnell], LCM) use stronger forms of **distillation** built on the
same idea: teach a student network to go in one step to where the teacher network goes in many steps.

## 10 · Diffusion in the autoencoder's codes: digits from noise

Now for MNIST. The diffusion model never touches a pixel: it generates 16-number codes, and Lesson 19's decoder draws
the codes as images. We use exactly the flow-matching network that the demo runs in the browser (H = 256, 40,000
steps, seed 0) and the same noise that `sweep.py` used to evaluate it (`default_rng(7)`, float32): all 10,000 noise
samples for 1 and 4 steps, the first 2,000 for the longer runs. The sweep results store the standard deviation of the
10,000 generated codes for each number of steps; we recompute that number to check our network.

Heun is a second-order solver: take a trial Euler step, measure the velocity at the landing point, then redo the step
with the average of the two velocities. Each step costs two network calls, except the last one (which lands on
$t = 0$ with Euler), so $k$ Heun steps cost $2k - 1$ calls.

In [ ]:
def heun(v, z, ts):
    x = z.copy()
    for a, b in itertools.pairwise(ts):
        va = v(x, a)
        if b > 0:
            va = 0.5 * (va + v(x - (a - b) * va, b))
        x = x - (a - b) * va
    return x


NET_FM = {k: unpack(t) for k, t in NETS["nets"]["fmB"]["p"].items()}
v_fm = net_v(NET_FM)
Z_NOISE = np.random.default_rng(7).standard_normal((10000, 16)).astype(np.float32).astype(np.float64)
print(f"the browser's network: H = {NETS['nets']['fmB']['H']}, {fmt(NETS['nets']['fmB']['steps'], 0)} training steps; "
      f"the sweep's networks: H = {MN['setup']['H']}, {fmt(MN['setup']['steps'], 0)} steps, 3 seeds")  # fmt: skip
t0 = time.perf_counter()
traj = []  # 32 Euler steps on the first 2,000 noise samples; keep every xₜ for the section below
codes = {
    "1 Euler step": (euler(v_fm, Z_NOISE, grid(1)), "euler", 1),
    "4 Euler steps": (euler(v_fm, Z_NOISE, grid(4)), "euler", 4),
    "32 Euler steps": (euler(v_fm, Z_NOISE[:2000], grid(32), traj), "euler", 32),
    "5 Heun steps": (heun(v_fm, Z_NOISE[:2000], grid(5)), "heun", 9),
}
print(f"sampling: {fmt(time.perf_counter() - t0, 1)} seconds")
for name, (Zs, smp, nfe) in codes.items():
    stored = cells("ae16-fmB", smp, nfe, "codeStd")[0]
    print(f"{name:>14} ({nfe:>2} network {'call, ' if nfe == 1 else 'calls,'} {fmt(len(Zs), 0)} samples): standard deviation of the codes {fmt(Zs.std(), 3)} "
          f"(demo, 10,000 samples: {fmt(stored, 3)}), stored FD {fmt(cells('ae16-fmB', smp, nfe)[0])}")  # fmt: skip
    assert abs(Zs.std() - stored) < (0.002 if len(Zs) == 10000 else 0.01), (
        "differs from the demo: check the network again"
    )

Our network gives exactly the numbers the demo stored, so this really is the demo's network. The figure below decodes
the first 16 codes of each run. The first panel decodes the noise $\mathcal N(0, I)$ itself (0 steps): that is how
Lesson 19 sampled, with the same decoder.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2.6))
show_tile(axes[0], decode(Z_NOISE[:16]), 4, 4, "decoding noise N(0, I)")
for ax, (name, (Zs, _, nfe)) in zip(axes[1:], codes.items()):
    show_tile(ax, decode(Zs[:16]), 4, 4, f"{name}\n({nfe} network {'call' if nfe == 1 else 'calls'})")
plt.tight_layout()
plt.show()


def spread(Z):
    """The mean distance from each code to the mean of the codes: how diverse the samples are."""
    return np.sqrt(((Z - Z.mean(0)) ** 2).sum(1)).mean()


print(
    f"spread of the codes: real codes {fmt(spread(Zc))} · "
    + " · ".join(f"{n} {fmt(spread(c[0]))}" for n, c in codes.items())
)
print(f"standard deviation around the samples' own mean: real codes {fmt(np.sqrt(((Zc - Zc.mean(0)) ** 2).mean()))}, "
      f"1 Euler step {fmt(np.sqrt(((codes['1 Euler step'][0] - codes['1 Euler step'][0].mean(0)) ** 2).mean()))}")  # fmt: skip
z1 = codes["1 Euler step"][0]
print(f"1 step: the mean of the samples lies {fmt(np.linalg.norm(z1.mean(0) - Zc.mean(0)))} from the real mean code; "
      f"the standard deviation of all the numbers in the codes is still {fmt(z1.std())}, because the mean code itself "
      f"has 16 numbers with a standard deviation of {fmt(Zc.mean(0).std())}")  # fmt: skip
assert spread(z1) < 0.2 * spread(Zc)

Decoding the noise directly gives scribbles: the decoder of an autoencoder without KL only knows how to draw codes that
lie near the real codes, while $\mathcal N(0, I)$ also covers the empty regions between them (Lesson 19). One Euler
step gives 16 nearly identical images: the spread of the codes is only 0.28, versus 3.38 for the real codes (standard
deviation around the samples' own mean: 0.08 versus 0.86). The "standard deviation of all the numbers in the codes"
statistic that the demo prints is still 0.52 here, only because the 16 numbers of the mean code itself already have
that standard deviation; it does not measure diversity. 4 steps already give readable but still blurry digits;
32 Euler steps or 5 Heun steps (9 network calls) give sharp ones.

### Broad strokes first, details later
At each step, the network has a predicted image $\hat x_0 = x_t - t\,v$ (section 7). The figure below follows 4
samples through 32 Euler steps: for each sample, the top row decodes $x_t$ and the bottom row decodes $\hat x_0$, at
the 9 times $t$ written above the columns.

In [ ]:
xt_all = [Z_NOISE[:2000], *traj]  # x_t at t = 1, 31/32, …, 0
ts32 = grid(32)
# Euler step x_{t−Δt} = x_t − Δt·v, so v = (x_t − x_{t−Δt})/Δt and x̂₀ = x_t − t·v: no need to call the network again
x0_hats = [xt_all[i] - ts32[i] * (xt_all[i] - xt_all[i + 1]) / (ts32[i] - ts32[i + 1]) for i in range(32)] + [traj[-1]]
pick = np.round(np.linspace(0, 32, 9)).astype(int)
fig, axes = plt.subplots(8, 9, figsize=(9, 8.6))
for r in range(4):
    for c, i in enumerate(pick):
        for row, src in ((2 * r, xt_all), (2 * r + 1, x0_hats)):
            axes[row, c].imshow(decode(src[i][r]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
            axes[row, c].set_xticks([])
            axes[row, c].set_yticks([])
        axes[0, c].set_title(f"t = {fmt(ts32[i], 2)}", fontsize=9)
    axes[2 * r, 0].set_ylabel("xₜ", fontsize=9)
    axes[2 * r + 1, 0].set_ylabel("x̂₀", fontsize=9)
plt.tight_layout()
plt.show()

$x_t$ decodes to a blurry blob until fairly late, but $\hat x_0$ takes the shape of a digit early on: at $t = 1$ it is
a faint shadow of the average digit, at $t = 0.75$ it already has the shape of a digit, though not necessarily the
final one (in the figure, some samples still change to another digit), by $t = 0.5$ it is readable, and the second
half of the trajectory only refines the strokes. We measure this with a simple classifier on the codes: the majority
digit among the 5 nearest training codes (it gets 90.8% of the real codes right when the code itself is left out). At
each step, the fraction of the 2,000 samples whose $\hat x_0$ already carries the digit of the final image:

In [ ]:
def knn_digit(Z, k=5, exclude_self=False):
    """The majority digit among the k nearest training codes (a tie goes to the smaller digit)."""
    d2 = (Z**2).sum(1)[:, None] - 2 * Z @ Zc.T + (Zc**2).sum(1)[None]
    if exclude_self:
        np.fill_diagonal(d2, np.inf)
    nn = np.argpartition(d2, k - 1, axis=1)[:, :k]
    return (y_img[nn][:, :, None] == np.arange(10)).sum(1).argmax(1)


judge_acc = (knn_digit(Zc, exclude_self=True) == y_img).mean()
final = knn_digit(x0_hats[-1])
same = {i: (knn_digit(x0_hats[i]) == final).mean() for i in pick}
demo_same = {round(r["t"], 4): r["same"] for r in MN["traj"]}
print(
    f"5-nearest-neighbor classifier on the codes: {pct(judge_acc, 1)} of the training codes right (leaving each one out)"
)
for i in pick:
    t = ts32[i]
    print(f"t = {fmt(t, 2)}: x̂₀ already carries the final digit {pct(same[i], 0):>4} "
          f"(demo, LeNet on the images: {pct(demo_same[round(t, 4)], 0)})")  # fmt: skip
assert same[16] > 0.75 and same[0] < 0.3

Two different classifiers (ours reads codes, the demo's LeNet reads images) tell the same story: at $t = 1$ only about
10% (a random guess among 10 digits), by $t = 0.5$ already 86%. The diffusion model decides **which digit** in the
first half, then uses the second half to draw the details. For real images, Dieleman (2024) calls this "spectral
autoregression": Gaussian noise wipes out the fine details (high frequencies) first, so denoising rebuilds the large
shapes first.

### How good is it? Reading the sweep results
FD measures the distance between 10,000 samples and 10,000 test images, through LeNet's features (Lessons 19–20; low
is good). Recall measures the fraction of real images that the samples "cover" (Lesson 20). Every row uses the same
decoder; only the way the codes are obtained differs:

In [ ]:
def base(space, kind, key="fd"):
    return [b[key] for b in sorted(MN["base"], key=lambda b: b["seed"]) if b["space"] == space and b["kind"] == kind]


floors = {sp: [f["fd"] for f in MN["floors"] if f["space"] == sp] for sp in ("ae16", "vae16")}
rows = [
    ("decoding the test images' own codes (floor)", floors["ae16"], None),
    ("flow matching, 5 Heun steps (9 calls)", cells("ae16-fm", "heun", 9), cells("ae16-fm", "heun", 9, "R")),
    ("flow matching, 100 Euler steps", cells("ae16-fm", "euler", 100), cells("ae16-fm", "euler", 100, "R")),
    ("flow matching, 32 Euler steps", cells("ae16-fm", "euler", 32), cells("ae16-fm", "euler", 32, "R")),
    ("10-Gaussian mixture fitted to the codes (Lesson 19)", base("ae16", "gmm"), base("ae16", "gmm", "R")),
    ("Lesson 20's GAN (generates pixels directly)", MN["ref"]["gan"], None),
    ("one Gaussian fitted to the codes", base("ae16", "gauss"), base("ae16", "gauss", "R")),
    ("N(0, I), as when decoding noise above", base("ae16", "prior"), base("ae16", "prior", "R")),
]
print(f"{'source of the codes (16-D autoencoder, 3 seeds)':<52} {'FD':>11}  recall")
for name, fd, rc in rows:
    print(f"{name:<52} {span(fd):>11}  {span(rc, 2) if rc else '—'}")
print(f"\nVAE codes (β = 1) instead of the autoencoder's: flow matching, 100 steps {span(cells('vae16-fm', 'euler', 100))}, "
      f"N(0, I) {span(base('vae16', 'prior'))}, floor {span(floors['vae16'])}")  # fmt: skip
print(f"two sets of real images: recall {span(MN['ref']['realR'], 2)}")

The same Lesson 19 decoder; only the source of the codes differs:
- Taking the codes from $\mathcal N(0, I)$ gives an FD of 75–94. A small flow-matching model that generates the codes
  gives 3.4–3.6 after 100 Euler steps, far better than the GAN of Lesson 20 (7.9–8.0) and the Gaussian mixture of
  Lesson 19 (7.0–7.6). The decoder's floor (decoding the test images' own codes) is 0.8–0.9, so most of the error
  that remains lies in the model of the codes, not in the decoder.
- 5 Heun steps (9 network calls) give an even lower FD than 100 Euler steps, but a slightly lower recall (0.76 versus
  0.78–0.79). FD lumps realism and coverage into a single number, so two ways of sampling with similar FDs can still
  differ in coverage.
- The VAE's codes are much closer to $\mathcal N(0, I)$ (FD 31.2–31.8 when decoding noise, versus 75–94), yet the
  diffusion model on the VAE codes does worse (5.3–6.0), because the VAE's decoder draws worse (floor 2.0–2.1). Once
  a diffusion model takes care of the distribution, the autoencoder only needs to compress well. That is also why
  Stable Diffusion chooses a very small KL ($10^{-6}$, Lesson 19).

## 11 · The optimal denoiser can only copy

In section 4, the optimal denoiser of the ring of 8 clusters used the **true distribution** of the data. In practice
we have only $n$ training images, and the true minimum of the loss is the optimal denoiser for the uniform
distribution on exactly those $n$ points:

$$\mathbb E[x_0 \mid x_t] = \sum_{i=1}^{n} w_i\, z_i, \qquad w = \mathrm{softmax}\Big(-\frac{\lVert x_t - (1 - t)\,z_i \rVert^2}{2t^2}\Big),$$

a softmax-weighted average of the training codes $z_i$. When $t$ is small, the softmax puts all the weight on the
nearest code.

In [ ]:
def exact_x0(xt, t, Z):
    """E[x₀ | xₜ] when x₀ is uniform over the rows of Z (xₜ = (1 − t)x₀ + tε)."""
    lw = -((xt**2).sum(1)[:, None] - 2 * (1 - t) * xt @ Z.T + (1 - t) ** 2 * (Z**2).sum(1)[None]) / (2 * t * t)
    w = np.exp(lw - lw.max(1, keepdims=True))
    return (w / w.sum(1, keepdims=True)) @ Z


def exact_sample(Z, z, steps=50):
    """Euler on the velocity (xₜ − x̂₀)/t of the optimal denoiser; the last step lands directly on x̂₀ (as in the demo)."""
    ts, x = grid(steps), z.copy()
    for a, b in itertools.pairwise(ts[:-1]):
        x = x - (a - b) * (x - exact_x0(x, a, Z)) / a
    return exact_x0(x, ts[-2], Z)

### 🤔 Predict first
Take 1,000 samples with the optimal denoiser for 1,000 training codes. How many of the samples are **copies** of a
training code, and how many distinct codes do they hit?

In [ ]:
memo = {e["n"]: e for e in MN["memo"]["exact"]}
S_exact = {}
for n in (100, 1000):
    S_exact[n] = exact_sample(Zc[:n], Z_NOISE[:1000])
    d, idx = nearest(S_exact[n], Zc[:n])
    expect = n * (1 - (1 - 1 / n) ** 1000)  # distinct codes when drawing 1,000 times uniformly, with replacement
    print(f"{fmt(n, 0):>5} training codes: largest distance to the nearest code {fmt(d.max(), 9)}, "
          f"{fmt(len(np.unique(idx)), 0)} distinct codes (demo: {fmt(memo[n]['distinct'], 0)}; uniform draws: {fmt(expect, 1)})")  # fmt: skip
    assert d.max() < 1e-6 and abs(len(np.unique(idx)) - memo[n]["distinct"]) <= 5, "differs from the demo"
print(
    "demo, every set size:",
    " · ".join(f"{fmt(e['n'], 0)} codes → {pct(e['copy'], 0)} copies" for e in MN["memo"]["exact"]),
)

Every sample coincides exactly with a training code (a distance of exactly 0), at every set size the demo tried, even
55,000 codes. With 1,000 codes, the 1,000 samples hit 637 distinct codes, close to the number of distinct codes you
get by drawing uniformly at random 1,000 times (632): the optimal denoiser is just a way of **picking a training image
at random**. The true minimum of the diffusion loss is memorization.

So why do real networks generate new digits? Because they **don't** reach that minimum: a small, smooth network with
limited training interpolates between the codes instead of memorizing each one (inductive bias). With little data, the
network is able to memorize everything. The cell below trains the same network as section 6 (now on the 16-number
codes) on 10 and on 500 codes, then counts the samples that "are copies": their distance to the nearest training code
is smaller than a level that 99% of unseen codes (the codes of the remaining 500 images) all exceed. That is how the
demo and Lesson 20 measure it.

In [ ]:
held = Zc[500:]  # 500 codes never used for training
z_copy = Z_NOISE[:500]
live_copy = {}
for n in (10, 500):
    t0 = time.perf_counter()
    p_n, _ = train_fm(Zc[:n], steps=1500, lr=3e-3)
    S_n = euler(net_v(p_n), z_copy, grid(32))
    thr = np.quantile(nearest(held, Zc[:n])[0], 0.01)  # 99% of the unseen codes are farther away than this
    live_copy[n] = (nearest(S_n, Zc[:n])[0] < thr).mean()
    print(
        f"trained on {fmt(n, 0):>3} codes ({fmt(time.perf_counter() - t0, 1)} seconds): {pct(live_copy[n], 1)} of the samples are copies"
    )
assert live_copy[10] > 0.4 and live_copy[500] < 0.05

Same network, same number of training steps: trained on 10 codes, most of the samples are copies; on 500 codes, almost
none are. The demo measures this with a large network (H = 512, 20,000 steps, 3 seeds) on 100 to 55,000 codes, and
also measures it in pixel space (the decoded samples compared with the MNIST training images themselves):

In [ ]:
print(
    f"{'training codes':>16} │ {'optimal denoiser':>16} │ {'network: copies (codes)':>23} │ {'(pixels)':>10} │ FD        │ recall"
)
for n in (100, 1000, 10000, 55000):
    model = "ae16-fm" if n == 55000 else f"ae16-n{n}"
    c = {k: cells(model, "euler", 100, k) for k in ("copy", "copyPix", "fd", "R")}
    print(f"{fmt(n, 0):>16} │ {pct(memo[n]['copy'], 0):>16} │ {span(c['copy'], 1, 100, '%'):>23} │ "
          f"{span(c['copyPix'], 1, 100, '%'):>10} │ {span(c['fd']):<9} │ {span(c['R'], 2)}")  # fmt: skip
print(
    f"Lesson 20's GAN on 1,000 images: {span(MN['ref']['ganCopyN1000'], 1, 100, '%')} of the samples are copies (pixels)"
)

Trained on 100 codes, 96.2–97.9% of the samples are copies of a training code, and recall is only 0.12–0.13: the model
copies its small training set. On 55,000 codes, only 0.1–0.2% are. At 1,000 images, 4.3–4.6% of the samples nearly
copy a training image when measured in pixels, versus 0.0–0.1% for the GAN of Lesson 20 with the same amount of data.
The same holds at large scale: Carlini et al. (2023) generated 500 images for each caption of the 350,000 most
duplicated images in Stable Diffusion's training data (175 million generations) and extracted 94 near-identical
training images; in their CIFAR-10 experiments, the diffusion models leaked more than twice as many training images as
the GANs. Kadkhodaie et al. (2023) show the other side: with enough data, two networks trained on two disjoint halves
of a set of images generate nearly the same image from the same noise, which means they learn the same distribution
rather than memorizing images.

## 12 · Classifier-free guidance: true to the label, or diverse

To generate a chosen digit, we give the network the label as well: $v_\theta(x_t, t, c)$. The demo trains **one** such
network, replacing the label with "no label" ∅ for 10% of the samples in each batch, so the same network is both a
conditional and an unconditional network. At sampling time, classifier-free guidance (Ho & Salimans, 2022)
extrapolates from the unlabeled prediction toward the labeled one:

$$\tilde v = v_\varnothing + s\,(v_c - v_\varnothing).$$

$s = 0$ means using only the unlabeled branch, $s = 1$ is the ordinary conditional network, and $s > 1$ is
"guidance": pushing further in the direction that makes the image look more like digit $c$. Each step costs two
network calls. The cell below runs the conditional network that the demo uses in the browser (32 Euler steps, 2,000
samples, 200 samples per digit) and checks the standard deviation of the codes against the sweep results. It measures
accuracy with the 5-nearest-neighbor classifier of section 10, and the precision/recall (Lesson 20) of 1,000 samples
against 1,000 real codes.

### 🤔 Predict first
Going from $s = 1$ up to $s = 3$, digit accuracy rises. What about recall, the fraction of real handwriting styles that
the samples cover: will it rise, stay the same or fall?

In [ ]:
NET_CFG = {k: unpack(t) for k, t in NETS["nets"]["cfgB"]["p"].items()}


def v_cfg(y, s):
    """The guided velocity with labels y (an array of 0–9) and scale s."""
    null = np.full(len(y), 10)

    def v(x, t):
        if s == 0:
            return forward(NET_CFG, inputs(x, t, null))
        vc = forward(NET_CFG, inputs(x, t, y))
        if s == 1:
            return vc
        vu = forward(NET_CFG, inputs(x, t, null))
        return vu + s * (vc - vu)

    return v


def prec_rec(real, fake, k=3):
    """Precision (samples inside the k-NN ball of a real point) and recall (real points inside the ball of a sample)."""

    def radius(A):
        d2 = np.maximum((A**2).sum(1)[:, None] - 2 * A @ A.T + (A**2).sum(1)[None], 0)
        np.fill_diagonal(d2, np.inf)
        return np.sqrt(np.partition(d2, k - 1, axis=1)[:, k - 1])

    D = np.sqrt(np.maximum((fake**2).sum(1)[:, None] - 2 * fake @ real.T + (real**2).sum(1)[None], 0))
    return (D <= radius(real)[None]).any(1).mean(), (D.T <= radius(fake)[None]).any(1).mean()


y_req = np.arange(2000) % 10  # the requested digits, as in the sweep results
guided = {}
for s_g in (0, 1, 3, 9):
    Zg = euler(v_cfg(y_req, s_g), Z_NOISE[:2000], grid(32))
    acc = (knn_digit(Zg) == y_req).mean()
    P, Rc = prec_rec(Zc, Zg[:1000])
    guided[s_g] = {"Z": Zg, "acc": acc, "P": P, "R": Rc}
    stored = cells("ae16-cfgB", "cfg", 32, "codeStd", s=s_g)[0]
    print(f"s = {s_g}: requested digit right {pct(acc, 1):>6} (5 nearest neighbors on the codes) · precision {fmt(P)} · recall {fmt(Rc)}"
          f" · standard deviation of the codes {fmt(Zg.std(), 3)} (demo: {fmt(stored, 3)})")  # fmt: skip
    assert abs(Zg.std() - stored) < 0.01, "differs from the demo: check the conditional network again"
P_real, R_real = prec_rec(Zc[:500], Zc[500:])
print(f"two halves of the 1,000 real codes: precision {fmt(P_real)}, recall {fmt(R_real)}")
assert guided[3]["acc"] > guided[1]["acc"] and guided[3]["R"] < guided[1]["R"] - 0.15

Our network gives exactly the standard deviation the demo stored for each $s$ (off only in the third decimal place,
because we take 2,000 samples instead of 10,000). The 5-nearest-neighbor classifier of section 10 sees accuracy rise
from 90.6% to 99.7% as $s$ goes from 1 to 3, while the recall on the codes falls from 0.86 to 0.55. The figure below
draws 16 samples of "digit 3" at each $s$, with the same noise (the titles give the accuracy over all 2,000 samples,
of every digit):

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, (s_g, g) in zip(axes, guided.items()):
    sel = np.flatnonzero(y_req == 3)[:16]
    title = "s = 0 (no label)" if s_g == 0 else f"s = {s_g}"
    show_tile(ax, decode(g["Z"][sel]), 4, 4, f"{title}: {pct(g['acc'], 0)} right")
plt.tight_layout()
plt.show()

Dropping the label ($s = 0$) gives all kinds of digits. $s = 1$ already gives mostly 3s, in every handwriting style.
$s = 3$ gives 3s that are all clear, bold and more alike. $s = 9$ pushes the codes far from the data's region (standard
deviation 1.58 versus ≈ 1): thick, distorted strokes, and precision collapses. The demo's sweep results (H = 512
network, 3 seeds, 1,000 samples per digit, accuracy read by LeNet on the images):

In [ ]:
SCALES = sorted({c["s"] for c in MN["cells"] if c["model"] == "ae16-cfg"})
print(f"{'s':>4} │   digit right │ precision │ recall    │ FD")
for s_g in SCALES:
    acc_s = cells("ae16-cfg", "cfg", 32, "acc", s=s_g)
    print(f"{tick(s_g):>4} │ {span(acc_s, 2, 100, '%') if acc_s else '—':>13} │ "
          f"{span(cells('ae16-cfg', 'cfg', 32, 'P', s=s_g), 2):>9} │ {span(cells('ae16-cfg', 'cfg', 32, 'R', s=s_g), 2):>9} │ "
          f"{span(cells('ae16-cfg', 'cfg', 32, 'fd', s=s_g))}")  # fmt: skip
print(f"unconditional network (section 10), 32 steps: FD {span(cells('ae16-fm', 'euler', 32))}, "
      f"recall {span(cells('ae16-fm', 'euler', 32, 'R'), 2)}")  # fmt: skip

fig, ax = plt.subplots(figsize=(7, 3.4))
for key, color, marker, ls, label in (("acc", BLUE, "o", "-", "requested digit right"), ("R", ORANGE, "s", "-", "recall"),
                                      ("P", GREEN, "D", "--", "precision")):  # fmt: skip
    ws = [w for w in SCALES if w > 0]
    vals = [cells("ae16-cfg", "cfg", 32, key, s=w) for w in ws]
    ax.plot(ws, [np.median(v) for v in vals], color=color, ls=ls, lw=1.8, marker=marker, ms=5, label=label)
    for w, v in zip(ws, vals):
        ax.scatter([w] * len(v), v, color=color, marker=marker, s=18, zorder=3)
ax.set_xscale("log")
ax.set(
    xlabel="guidance scale s (log scale)",
    ylabel="fraction",
    ylim=(0, 1.02),
    title="Guidance: closer to the label, less diversity",
)
ax.legend(frameon=False, loc="lower left")
fmt_axes(fig)
plt.show()

The blue line (round dots) is accuracy, the orange line (squares) recall, the dashed green line (diamonds) precision;
each marker is one seed. Reading the table and the figure:
- **Conditioning is not yet guidance.** The conditional network ($s = 1$) already generates the requested digit
  96.48–96.67% of the time, and its FD of 1.8–1.9 is already better than the unconditional network's (3.7–3.9 at the
  same 32 steps).
- **A little guidance helps, a lot hurts.** FD is best at $s = 1.1$ (1.3–1.4), then rises steadily. Ho & Salimans
  (2022), who write $w = s - 1$, also find the best FID with $w$ = 0.1 or 0.3.
- **Diversity traded for adherence.** With $s = 3$, accuracy rises to 99.99%, but recall falls from 0.75–0.76 to
  0.39–0.41 and FD rises to 8.3–9.0: every sample is a "typical" digit, and the rare handwriting styles disappear.
- **Overdo it and accuracy suffers too.** Precision reaches 0.94–0.95 at $s$ = 2–3 and then falls, down to 0.74–0.82
  at $s = 9$, where the codes have drifted well out of the data's region and accuracy also starts to fall
  (97.97–99.77%).

Real image models usually use larger scales: the Stable Diffusion 1.x pipeline defaults to 7.5, most experiments in
the Stable Diffusion 3 paper run at 5, and the FLUX.1 [dev] examples use 3.5 (that network is guidance-distilled, so
each step costs only one network call).

## 13 · Try changing things

The cells below rerun the same functions with other parameters. You can edit the values and run them again.

### Narrower clusters need more steps
Change the width of each cluster of the ring from 0.1 to 0.02 ("sharper" data), still with the optimal denoiser.
Change `S_TRY` or `STEPS_TRY` and run again.

In [ ]:
S_TRY, STEPS_TRY = 0.02, (2, 4, 8, 32)
for k in STEPS_TRY:
    q_eu = [ring_quality(euler(lambda x, t: ring_v(x, t, S_TRY), z, grid(k)), S_TRY)[0] for z in Z_DEMO]
    q_he = [
        ring_quality(heun(lambda x, t: ring_v(x, t, S_TRY), z, grid(max(1, (k + 1) // 2))), S_TRY)[0] for z in Z_DEMO
    ]
    print(f"{k:>2} call{'s' if k > 1 else ''}: Euler {span(q_eu, 1, 100, '%'):>12} · Heun ({max(1, (k + 1) // 2)} step{'s' if k > 2 else ''}, "
          f"{2 * max(1, (k + 1) // 2) - 1} call{'s' if k > 2 else ''}) {span(q_he, 1, 100, '%')}")  # fmt: skip
    if S_TRY == 0.02:
        assert np.allclose(q_eu, [r["inMode"] for r in RING2D["table"]["0.02"]["rows"][f"euler/{k}"]], atol=1e-3)

With clusters of width 0.02, 4 Euler steps put only 57.5–59.6% of the samples in a cluster (versus 76.8–78.4% at
width 0.1): the sharper the data, the more sharply the velocity field turns near the end of the path, and the more
easily a big step slides off the cluster. Heun, the second-order solver, is here actually **worse** than Euler at the
same small number of network calls (89.9–91.8% with 7 calls, versus 98.2–98.7% for 8 Euler steps): a higher-order
solver only pays off when each step is small enough for the field to be nearly smooth within it. On the MNIST codes,
where the field is smoother, Heun wins again (section 10: 9 calls give an FD of 2.9–3.2).

### Guidance for another digit
Change `DIGIT` and `S_LIST`. The cell below prints the accuracy (the 5-nearest-neighbor classifier of section 10) and
the spread of the codes over 200 samples.

In [ ]:
DIGIT, S_LIST = 8, (1, 2, 4)
z200, y200 = Z_NOISE[2000:2200], np.full(200, DIGIT)
real_spread = spread(Zc[y_img == DIGIT])
for s_g in S_LIST:
    Zd = euler(v_cfg(y200, s_g), z200, grid(32))
    print(f"digit {DIGIT}, s = {tick(s_g)}: {pct((knn_digit(Zd) == DIGIT).mean(), 1)} right, spread of the codes "
          f"{fmt(spread(Zd))} (real codes of digit {DIGIT}: {fmt(real_spread)})")  # fmt: skip

As $s$ grows, the spread of the samples' codes usually shrinks relative to the real codes of the same digit: the
samples are more alike. Try $s = 0.5$ (between unlabeled and labeled) to see the opposite direction.

### More data for the network you train yourself
In section 11, change `(10, 500)` to `(10, 50, 100, 500)` to see the copy rate fall as the number of training codes
grows. In section 6, change `train_fm(X_ring)` to `train_fm(X_ring, steps=1000)` and rerun section 6: a network that
has learned less well needs more Euler steps to get into the clusters.

## 14 · Limits and next steps

- **A miniature model.** Here the denoising network is an MLP on 16 numbers. Stable Diffusion denoises a 64×64×4 grid
  of codes (SD3: 16 channels) with a U-Net, then with a Transformer (DiT, SD3's MM-DiT, FLUX.1 with 12 billion
  parameters), and reads the text prompt through attention (Lessons 16–17) instead of a one-hot label.
- **Still many network calls.** 32 steps is few compared with 1,000, but it is still 32 runs of the whole network for
  one image (64 with guidance). Reflow and distillation (consistency models, SDXL Turbo, FLUX.1 [schnell]) bring that
  number down to 1–4.
- **The course's metric.** FD here is computed on the features of LeNet (Lesson 12), so it can only be compared within
  the course, not with the FID of papers.
- **Memorization.** Section 11 shows that the minimum of the loss is copying; real networks generalize only with
  enough data.

The text that Stable Diffusion reads goes through CLIP's text encoder, a model **pretrained** on hundreds of millions
of image–caption pairs, and users often **fine-tune** the diffusion model itself with LoRA to teach it a new style.
That is the subject of the next lesson.

## Summary
- The forward process needs no learning: $x_t = (1 - t)\,x_0 + t\,\varepsilon$, available directly at any $t$. The
  network learns the velocity $\varepsilon - x_0$ with squared error; the loss doesn't go to 0 (a floor of 3.59 on the
  ring of 8 clusters), because $x_t$ doesn't tell which pair it came from.
- Predicting $\varepsilon$, predicting $x_0$ and predicting $v$ are interchangeable ($\hat x_0 = x_t - t\,v$), differing
  only in the weighting over $t$; each has its own blind spot (the network that predicts $\varepsilon$ has an error of
  92–97 at $t = 0.98$, versus 12–13).
- One DDIM step is one Euler step; DDPM, DDIM and flow matching differ in where the steps are placed. With the
  optimal denoiser and 4 steps: the uniform grid in $t$ puts 76.8–78.4% of the samples in a cluster, DDPM's grid
  1.8–1.9%.
- The number of steps is a choice of the solver: 32 Euler steps give an FD of 3.7–3.9, 1,000 steps 3.45. One step
  falls to the mean (FD 247–284); reflow straightens the paths, and one step then gives 5.4–5.6.
- Diffusion in the codes of Lesson 19's autoencoder: FD 3.4–3.6, versus 75–94 when decoding noise
  $\mathcal N(0, I)$ and 7.9–8.0 for Lesson 20's GAN.
- The optimal denoiser for a finite set only copies (100% copies); a real network copies 96.2–97.9% when trained on
  100 codes, 0.1–0.2% when trained on 55,000.
- Classifier-free guidance: $s = 3$ gives 99.99% digit accuracy, but recall 0.75–0.76 → 0.39–0.41.

## Review questions
1. Why does one Euler step from $t = 1$, even with a perfect denoiser, send every sample to the same point? What is
   that point?
2. The flow-matching loss of the network doesn't go to 0 even when the network is perfect. Why, and what does that say
   about comparing the losses of two models?
3. A network that predicts $\varepsilon$ and a network that predicts $v$ learn the same denoiser. Why does the network
   that predicts $\varepsilon$ give much worse images after 100 Euler steps?
4. DDPM trains with $T = 1{,}000$. Does sampling also need 1,000 steps? Quote two numbers from the notebook.
5. The optimal denoiser for 55,000 training codes can be computed exactly. Why do people still train a network?
6. Raising the guidance scale $s$ from 1 to 3 makes accuracy rise and recall fall. What do these two numbers measure,
   and why do they move in opposite directions?

<details><summary>Hints and answers</summary>

1. At $t = 1$, $x_t$ is pure noise and carries no information about $x_0$, so $\mathbb E[x_0 \mid x_1]$ is the mean of
   all the data. An Euler step of size 1 goes straight to that $\hat x_0$: the origin for the ring of 8 clusters, the
   "average digit" for MNIST.
2. Many pairs $(x_0, \varepsilon)$ give the same $x_t$; the network can only predict the average of the targets
   $\varepsilon - x_0$, and the rest is unavoidable error (3.59 on the ring of 8 clusters). That floor depends on the
   data and on how $t$ is chosen, so the loss doesn't tell you which model generates better images.
3. Converting $\hat\varepsilon$ into a velocity requires dividing by $1 - t$; near $t = 1$, a small error in
   $\hat\varepsilon$ becomes a very large velocity error (92–97 at $t = 0.98$), precisely in the first steps, which
   decide the digit.
4. No. Ho et al. chose 1,000 to match previous work, without a sweep. 32 Euler steps give an FD of 3.7–3.9, versus
   3.45 at 1,000 steps; DDIM on the DDPM network with 100 steps gives 4.1–4.3.
5. Because it only copies the training images (100% copies). We want new images; the network generates new images
   precisely because it does not reach that minimum, thanks to inductive bias and enough data.
6. Accuracy measures whether a sample has the requested digit (adherence); recall measures how many real handwriting
   styles the samples cover (diversity). Guidance pushes each sample toward the "most typical" form of its label, so
   the samples are more often right but more alike.
</details>

## Further reading
- J. Ho, A. Jain & P. Abbeel (2020), [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239).
- J. Song, C. Meng & S. Ermon (2020), [DDIM](https://arxiv.org/abs/2010.02502): deterministic sampling in few steps,
  with the same DDPM network.
- Y. Lipman et al. (2022), [Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747); X. Liu, C. Gong &
  Q. Liu (2022), [Rectified Flow and reflow](https://arxiv.org/abs/2209.03003).
- [Diffusion Meets Flow Matching: Two Sides of the Same Coin](https://diffusionflow.github.io/) (a Google DeepMind
  blog post, ICLR 2025): DDIM = flow matching's Euler, and why the "straight" paths are straight only for each pair.
- D. Kingma & R. Gao (2023), [Understanding Diffusion Objectives](https://arxiv.org/abs/2303.00848): ε, x₀ and v differ
  only in their weighting.
- T. Karras et al. (2022), [EDM: Elucidating the Design Space of Diffusion-Based Generative Models](https://arxiv.org/abs/2206.00364).
- R. Rombach et al. (2022), [Latent Diffusion](https://arxiv.org/abs/2112.10752); P. Esser et al. (2024),
  [Stable Diffusion 3](https://arxiv.org/abs/2403.03206); W. Peebles & S. Xie (2022), [DiT](https://arxiv.org/abs/2212.09748).
- J. Ho & T. Salimans (2022), [Classifier-Free Diffusion Guidance](https://arxiv.org/abs/2207.12598).
- N. Carlini et al. (2023), [Extracting Training Data from Diffusion Models](https://arxiv.org/abs/2301.13188);
  Z. Kadkhodaie et al. (2023), [Generalization in diffusion models arises from geometry-adaptive harmonic representations](https://arxiv.org/abs/2310.02557).
- S. Dieleman (2024), [Diffusion is spectral autoregression](https://sander.ai/2024/09/02/spectral-autoregression.html).
- P. Holderrieth & E. Erives (2025), [MIT 6.S184: An Introduction to Flow Matching and Diffusion Models](https://arxiv.org/abs/2506.02070).

**Next lesson:** [Lesson 22 · Pretraining & fine-tuning](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/tien-huan-luyen/):
learn first from unlabeled data, then adjust a little for the task at hand.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")